<a href="https://colab.research.google.com/github/B00RR/generatore-buoni/blob/main/Sostituzione_QR_su_file_WORD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Installa le librerie necessarie
!pip install python-docx cairosvg -q
print("✅ Librerie installate: python-docx, cairosvg")


✅ Librerie installate: python-docx, cairosvg


In [11]:
from google.colab import files

print("📄 STEP 1: Carica il template Word (.docx)")
print("Seleziona il file BUONO-CARBURANTE-ELETTRONICO.docx\n")
template = files.upload()

print("\n📦 STEP 2: Carica il file ZIP con i buoni")
print("Seleziona il tuo file .zip contenente i buoni\n")
zip_buoni = files.upload()

print("\n✅ File caricati con successo!")
print("Ora esegui la CELLA 3 per convertire i QR SVG→PNG")


📄 STEP 1: Carica il template Word (.docx)
Seleziona il file BUONO-CARBURANTE-ELETTRONICO.docx



Saving BUONO CARBURANTE ELETTRONICO 30.docx to BUONO CARBURANTE ELETTRONICO 30.docx

📦 STEP 2: Carica il file ZIP con i buoni
Seleziona il tuo file .zip contenente i buoni



Saving qrcodes-52647-1761292631.zip to qrcodes-52647-1761292631 (2).zip

✅ File caricati con successo!
Ora esegui la CELLA 3 per convertire i QR SVG→PNG


In [12]:
# ═══════════════════════════════════════════════════════════════════════
# CELLA 3: Conversione SVG → PNG mantenendo struttura cartelle
# ═══════════════════════════════════════════════════════════════════════

import os
import re
from zipfile import ZipFile
import cairosvg
import shutil

# Pulizia cartelle precedenti
for folder in ['qr_convertiti', 'buoni_generati']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"🗑️  Eliminata cartella vecchia: {folder}")

if os.path.exists('buoni_carburante_generati.zip'):
    os.remove('buoni_carburante_generati.zip')
    print(f"🗑️  Eliminato ZIP vecchio")

print("\n" + "=" * 75)
print("CONVERSIONE QR CODE: SVG → PNG")
print("=" * 75)

# Trova il file ZIP caricato
zip_files = [f for f in os.listdir('.') if f.endswith('.zip') and f != 'buoni_carburante_generati.zip']

if not zip_files:
    print("\n❌ Errore: File ZIP non trovato!")
    print("   Ricarica il file nella CELLA 2")
else:
    zip_path = zip_files[0]
    print(f"\n📦 File ZIP: {zip_path}")

    # Crea cartella output per i PNG
    output_folder = 'qr_convertiti'
    os.makedirs(output_folder, exist_ok=True)

    svg_count = 0
    png_count = 0
    errori = 0

    print("\n🔄 Inizio conversione...\n")

    with ZipFile(zip_path, 'r') as zip_file:
        all_files = zip_file.namelist()

        for file_path in all_files:
            # Cerca solo file SVG
            if file_path.lower().endswith('.svg'):
                svg_count += 1

                # Estrai il nome della cartella principale (con il numero del buono)
                parts = file_path.split('/')
                if len(parts) >= 1:
                    folder_principale = parts[0]

                    # Estrai il numero del buono
                    match = re.search(r'- ([a-zA-Z0-9]+) -', folder_principale)
                    if match:
                        numero_buono = match.group(1)
                    else:
                        match = re.search(r'-([a-zA-Z0-9]+)-', folder_principale)
                        numero_buono = match.group(1) if match else f"unknown_{svg_count}"

                    try:
                        # Estrai SVG temporaneo
                        svg_temp = f'temp_{numero_buono}.svg'
                        with zip_file.open(file_path) as source:
                            with open(svg_temp, 'wb') as target:
                                target.write(source.read())

                        # Converti in PNG
                        png_output = os.path.join(output_folder, f'qr_{numero_buono}.png')
                        cairosvg.svg2png(url=svg_temp, write_to=png_output)

                        # Verifica dimensione
                        size = os.path.getsize(png_output)
                        print(f"  ✓ Buono {numero_buono}: {size} bytes")

                        png_count += 1

                        # Elimina SVG temporaneo
                        os.remove(svg_temp)

                    except Exception as e:
                        print(f"  ✗ Errore buono {numero_buono}: {e}")
                        errori += 1

    print("\n" + "=" * 75)
    print(f"✅ CONVERSIONE COMPLETATA!")
    print(f"   SVG trovati: {svg_count}")
    print(f"   PNG creati: {png_count}")
    if errori > 0:
        print(f"   Errori: {errori}")
    print("=" * 75)

    print(f"\n📁 QR PNG salvati in: {output_folder}/")
    print("\nOra esegui la CELLA 4 per generare i buoni!")


🗑️  Eliminata cartella vecchia: qr_convertiti
🗑️  Eliminata cartella vecchia: buoni_generati

CONVERSIONE QR CODE: SVG → PNG

📦 File ZIP: qrcodes-52647-1761292631 (1).zip

🔄 Inizio conversione...

  ✓ Buono e1aad: 7930 bytes
  ✓ Buono 86f7b: 7670 bytes
  ✓ Buono 293b3: 7915 bytes
  ✓ Buono 7ad9f: 7717 bytes
  ✓ Buono 28349: 7662 bytes
  ✓ Buono dfb11: 7879 bytes
  ✓ Buono c2400: 7841 bytes
  ✓ Buono df6c5: 7905 bytes
  ✓ Buono 4ceb9: 7743 bytes
  ✓ Buono 27a56: 7836 bytes
  ✓ Buono fc743: 7976 bytes
  ✓ Buono 8844d: 7914 bytes

✅ CONVERSIONE COMPLETATA!
   SVG trovati: 12
   PNG creati: 12

📁 QR PNG salvati in: qr_convertiti/

Ora esegui la CELLA 4 per generare i buoni!


In [13]:
import os
import re
from docx import Document
from docx.shared import Inches
from docx.oxml.ns import qn
from zipfile import ZipFile, ZIP_DEFLATED
import shutil

print("=" * 75)
print("GENERAZIONE BUONI - SOSTITUZIONE QR PARAGRAFO 5")
print("=" * 75)

# Pulizia di file temporanei e cartelle create in precedenza
print("\n🧹 Pulizia...")
for folder in ['buoni_generati']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"🗑️  Eliminata cartella vecchia: {folder}")

for file in os.listdir('.'):
    if file.startswith('temp_') and file.endswith('.docx'):
        try:
            os.remove(file)
        except:
            pass

# --- NUOVA LOGICA: ACCESSO DIRETTO ALLE VARIABILI DI CARICAMENTO ---

# Verifica se le variabili 'template' e 'zip_buoni' esistono (dalla CELLA 2)
if 'template' not in globals():
    print("\n❌ Variabile 'template' non trovata! Assicurati di aver eseguito la CELLA 2 per caricare il template Word.")
elif 'zip_buoni' not in globals():
    print("\n❌ Variabile 'zip_buoni' non trovata! Assicurati di aver eseguito la CELLA 2 per caricare il file ZIP.")
elif not os.path.exists('qr_convertiti'):
    print("\n❌ Esegui prima CELLA 3 per convertire i QR!")
else:
    # Estrai il nome e il contenuto del template caricato dalla variabile 'template'
    template_filename = list(template.keys())[0]
    template_content = list(template.values())[0]
    template_path = template_filename # Usiamo il nome originale come percorso temporaneo

    # Estrai il nome e il contenuto del file ZIP caricato dalla variabile 'zip_buoni'
    zip_filename = list(zip_buoni.keys())[0]
    zip_content = list(zip_buoni.values())[0]
    zip_path = zip_filename # Usiamo il nome originale come percorso temporaneo

    # Usa un blocco try-finally per garantire che i file temporanei vengano puliti
    try:
        # Scrivi il contenuto del template su un file locale temporaneo
        with open(template_path, 'wb') as f:
            f.write(template_content)
        # Scrivi il contenuto dello zip su un file locale temporaneo
        with open(zip_path, 'wb') as f:
            f.write(zip_content)

        print(f"\n✅ Template selezionato: {template_path}")
        print(f"✅ ZIP selezionato: {zip_path}")

        output_folder = "buoni_generati"
        os.makedirs(output_folder, exist_ok=True)

        # Trova buoni dal ZIP
        print("\n🔍 Analisi dei QR nel file ZIP...\n")

        vouchers = []
        with ZipFile(zip_path, 'r') as zf:
            for fp in zf.namelist():
                if fp.lower().endswith('.svg'):
                    folder = fp.split('/')[0]

                    match = re.search(r'- ([a-zA-Z0-9]+) -', folder)
                    if match:
                        num = match.group(1)
                    else:
                        match = re.search(r'-([a-zA-Z0-9]+)-', folder)
                        num = match.group(1) if match else None

                    if num and not any(v['number'] == num for v in vouchers):
                        vouchers.append({'number': num})
                        print(f"  ✓ Trovato buono {num}")

        if not vouchers:
            print("\n❌ Nessun buono SVG trovato nel file ZIP!")
        else:
            print(f"\n✅ Trovati {len(vouchers)} buoni nel ZIP.\n")

            # Funzione per sostituire QR
            def sostituisci_qr_paragrafo_5(doc, qr_png_path):
                """Sostituisce il QR al paragrafo 5"""
                if len(doc.paragraphs) <= 5:
                    return False

                para = doc.paragraphs[5]

                has_drawing = False
                for run in para.runs:
                    drawings = run._element.findall('.//' + qn('w:drawing'))
                    if drawings:
                        has_drawing = True
                        break

                if has_drawing:
                    for run in list(para.runs):
                        run._element.getparent().remove(run._element)

                    new_run = para.add_run()
                    new_run.add_picture(qr_png_path, width=Inches(1.5))
                    return True

                return False

            # Genera buoni
            print("📝 Generazione documenti Word...\n")

            success = 0
            failed = 0

            for idx, voucher in enumerate(vouchers, 1):
                num = voucher['number']
                print(f"[{idx}/{len(vouchers)}] Buono {num}")

                qr_png = f'qr_convertiti/qr_{num}.png'

                if not os.path.exists(qr_png):
                    print(f"    ❌ QR PNG per il buono {num} non trovato nella cartella 'qr_convertiti'. Assicurati che la CELLA 3 sia stata eseguita correttamente.")
                    failed += 1
                    continue

                try:
                    # Carica template
                    doc = Document(template_path)

                    # Sostituisci numero del buono nel testo (se presente un placeholder)
                    # Questa logica cerca 'Buono n. XXXXX' e lo sostituisce, mantiene il testo 'valido fino al'
                    # Attenzione: il template deve avere 'Buono n.' seguito da un placeholder e poi 'valido'
                    numero_sostituito = False
                    for para in doc.paragraphs:
                        if 'Buono n.' in para.text:
                            orig = para.text
                            if 'valido' in orig:
                                parti = orig.split('Buono n.')
                                if len(parti) >= 2:
                                    dopo = parti[1]
                                    # Trova la parte dopo 'valido' per ricomporre la frase
                                    valido_part_index = dopo.lower().find('valido')
                                    if valido_part_index != -1:
                                        testo_dopo_valido = dopo[valido_part_index + len('valido'):]
                                        nuovo = f"Buono n. {num} valido" + testo_dopo_valido

                                        # Svuota tutti i run del paragrafo e aggiungi il nuovo testo
                                        for run in para.runs:
                                            run.text = ''
                                        if para.runs:
                                            # Se il paragrafo aveva run, usa il primo per il testo
                                            para.runs[0].text = nuovo
                                        else:
                                            # Altrimenti, imposta il testo direttamente sul paragrafo
                                            para.text = nuovo
                                        numero_sostituito = True
                                        print(f"    → Numero sostituito con: {num}")
                                        break # Esci dal loop dei paragrafi dopo la sostituzione
                    if not numero_sostituito:
                        print(f"    ⚠ Nessun placeholder 'Buono n.' con 'valido' trovato per la sostituzione del numero del buono {num}")

                    # Sostituisci QR (al paragrafo 5)
                    if sostituisci_qr_paragrafo_5(doc, qr_png):
                        output = os.path.join(output_folder, f'Buono_{num}.docx')
                        doc.save(output)

                        qr_size = os.path.getsize(qr_png)
                        print(f"    → QR inserito: {qr_size} bytes")
                        print(f"    ✅ OK\n")
                        success += 1
                    else:
                        print(f"    ⚠ QR non sostituito nel paragrafo 5 per il buono {num}. Controlla la struttura del template.")
                        failed += 1

                except Exception as e:
                    print(f"    ❌ Errore durante la generazione del buono {num}: {e}\n")
                    failed += 1

            # RIEPILOGO FINALE
            print("=" * 75)
            print(f"✅ GENERAZIONE COMPLETATA: {success}/{len(vouchers)} buoni")
            if failed > 0:
                print(f"   Falliti: {failed}")
            print("=" * 75)

            print(f"\n📁 I buoni generati sono salvati in: {output_folder}/")
            print(f"   Totale file Word creati: {len(os.listdir(output_folder))}\n")
            print("➡️  Esegui la CELLA 5 per scegliere il formato di output e scaricare")

    finally:
        # Cleanup dei file temporanei scritti localmente all'inizio della cella
        if os.path.exists(template_path):
            os.remove(template_path)
            print(f"🗑️  Eliminato file temporaneo: {template_path}")
        if os.path.exists(zip_path):
            os.remove(zip_path)
            print(f"🗑️  Eliminato file temporaneo: {zip_path}")

GENERAZIONE BUONI - SOSTITUZIONE QR PARAGRAFO 5

🧹 Pulizia...

✅ Template selezionato: BUONO CARBURANTE ELETTRONICO 30.docx
✅ ZIP selezionato: qrcodes-52647-1761292631 (2).zip

🔍 Analisi dei QR nel file ZIP...

  ✓ Trovato buono e1aad
  ✓ Trovato buono 86f7b
  ✓ Trovato buono 293b3
  ✓ Trovato buono 7ad9f
  ✓ Trovato buono 28349
  ✓ Trovato buono dfb11
  ✓ Trovato buono c2400
  ✓ Trovato buono df6c5
  ✓ Trovato buono 4ceb9
  ✓ Trovato buono 27a56
  ✓ Trovato buono fc743
  ✓ Trovato buono 8844d

✅ Trovati 12 buoni nel ZIP.

📝 Generazione documenti Word...

[1/12] Buono e1aad
    → Numero sostituito con: e1aad
    → QR inserito: 7930 bytes
    ✅ OK

[2/12] Buono 86f7b
    → Numero sostituito con: 86f7b
    → QR inserito: 7670 bytes
    ✅ OK

[3/12] Buono 293b3
    → Numero sostituito con: 293b3
    → QR inserito: 7915 bytes
    ✅ OK

[4/12] Buono 7ad9f
    → Numero sostituito con: 7ad9f
    → QR inserito: 7717 bytes
    ✅ OK

[5/12] Buono 28349
    → Numero sostituito con: 28349
    → QR

In [16]:
# ═══════════════════════════════════════════════════════════════════════
# CELLA 5: CONVERSIONE FORMATO OUTPUT (solo PDF e ZIP)
# ═══════════════════════════════════════════════════════════════════════

import os
from zipfile import ZipFile, ZIP_DEFLATED
import subprocess

print("=" * 75)
print("CONVERSIONE FORMATO OUTPUT")
print("=" * 75)

output_folder = "buoni_generati"

if not os.path.exists(output_folder):
    print("\n❌ Errore: Cartella 'buoni_generati' non trovata!")
    print("   Esegui prima la CELLA 4\n")
else:
    buoni_files = [f for f in os.listdir(output_folder) if f.endswith('.docx')]

    if not buoni_files:
        print("\n❌ Errore: Nessun buono trovato!")
        print("   Esegui prima la CELLA 4\n")
    else:
        print(f"\n✅ Trovati {len(buoni_files)} buoni generati\n")

        # SCELTA FORMATO
        print("📄 SCEGLI IL FORMATO DI OUTPUT:\n")
        print("1. PDF unico (tutti i buoni in un PDF) ⭐ CONSIGLIATO PER STAMPA")
        print("2. Word separati (ZIP con tutti i .docx singoli)\n")

        scelta = input("Inserisci il numero (1 o 2): ").strip()

        while scelta not in ['1', '2']:
            print("❌ Scelta non valida!")
            scelta = input("Inserisci 1 o 2: ").strip()

        print()

        # ═══════════════════════════════════════════════════════════════
        # OPZIONE 1: PDF UNICO
        # ═══════════════════════════════════════════════════════════════

        if scelta == '1':
            print("📄 Creazione PDF unico...\n")

            try:
                print("📦 Verifica LibreOffice...")

                result = subprocess.run(['which', 'libreoffice'],
                                      capture_output=True, text=True)

                if result.returncode != 0:
                    print("📦 Installazione LibreOffice (può richiedere 1-2 minuti)...")
                    subprocess.run(['apt-get', 'update', '-qq'], check=True)
                    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libreoffice'],
                                 check=True)
                    print("✅ Installato\n")
                else:
                    print("✅ Presente\n")

                print("🔄 Conversione Word → PDF con LibreOffice...\n")
                pdf_files = []

                for idx, docx_file in enumerate(sorted(buoni_files), 1):
                    docx_path = os.path.join(output_folder, docx_file)
                    pdf_filename = docx_file.replace('.docx', '.pdf')

                    print(f"  [{idx}/{len(buoni_files)}] {docx_file}...", end=' ')

                    try:
                        result = subprocess.run([
                            'libreoffice',
                            '--headless',
                            '--convert-to', 'pdf',
                            '--outdir', output_folder,
                            docx_path
                        ], capture_output=True, text=True, timeout=30)

                        pdf_path = os.path.join(output_folder, pdf_filename)

                        if os.path.exists(pdf_path):
                            pdf_files.append(pdf_path)
                            print("✓")
                        else:
                            print("✗")

                    except Exception as e:
                        print(f"✗")

                if pdf_files:
                    print(f"\n📦 Installazione PyPDF2...")
                    subprocess.run(['pip', 'install', 'PyPDF2', '-q'], check=True)

                    from PyPDF2 import PdfMerger

                    print(f"📑 Unione di {len(pdf_files)} PDF in un unico file...")

                    merger = PdfMerger()
                    for pdf in sorted(pdf_files):
                        merger.append(pdf)

                    output_pdf = "buoni_carburante_COMPLETI.pdf"
                    merger.write(output_pdf)
                    merger.close()

                    print(f"\n✅ PDF creato: {output_pdf}")
                    print(f"   Dimensione: {os.path.getsize(output_pdf) / 1024:.1f} KB")
                    print(f"   Pagine: {len(pdf_files) * 2}")  # 2 pagine per buono
                    print(f"   Buoni: {len(pdf_files)}")

                    from google.colab import files
                    print("\n⬇️ Download automatico...")
                    files.download(output_pdf)

                    print("\n🎉 PDF PRONTO PER LA STAMPA!")
                    print("   Tutti i buoni in sequenza con QR corretti")
                else:
                    print("\n❌ Nessun PDF creato!")
                    print("💡 Prova l'opzione 2 (Word separati)")

            except Exception as e:
                print(f"\n❌ Errore: {e}")
                print("💡 Prova l'opzione 2 (Word separati)")

        # ═══════════════════════════════════════════════════════════════
        # OPZIONE 2: WORD SEPARATI (ZIP)
        # ═══════════════════════════════════════════════════════════════

        elif scelta == '2':
            print("📦 Creazione ZIP con file Word separati...\n")

            zip_file = "buoni_carburante_generati.zip"

            with ZipFile(zip_file, 'w', ZIP_DEFLATED) as zipf:
                for idx, docx_file in enumerate(sorted(buoni_files), 1):
                    file_path = os.path.join(output_folder, docx_file)
                    print(f"  [{idx}/{len(buoni_files)}] {docx_file}...", end=' ')
                    zipf.write(file_path, docx_file)
                    print("✓")

            print(f"\n✅ ZIP creato: {zip_file}")
            print(f"   Dimensione: {os.path.getsize(zip_file) / 1024:.1f} KB")
            print(f"   Contiene: {len(buoni_files)} file Word separati")
            print("   Ogni buono ha il suo QR unico corretto")

            from google.colab import files
            print("\n⬇️ Download automatico...")
            files.download(zip_file)

            print("\n🎉 ZIP SCARICATO!")
            print("   Estrai lo ZIP e avrai tutti i buoni pronti")

        print("\n" + "=" * 75)
        print("✅ OPERAZIONE COMPLETATA!")
        print("=" * 75)

        print("\n💡 SUGGERIMENTI:")
        if scelta == '1':
            print("   • Apri il PDF e stampa tutto")
            print("   • I QR sono unici e corretti per ogni buono")
        else:
            print("   • Estrai il ZIP sul tuo computer")
            print("   • Ogni file .docx è un buono separato")
            print("   • Puoi modificarli individualmente se serve")

        # Messaggio aggiuntivo richiesto dall'utente
        print("\n" + "*" * 75)
        print("⭐ Se hai bisogno di altri buoni, parti dalla CELLA 2, non c'è bisogno di cancellare gli output! ⭐")
        print("*" * 75)


CONVERSIONE FORMATO OUTPUT

✅ Trovati 12 buoni generati

📄 SCEGLI IL FORMATO DI OUTPUT:

1. PDF unico (tutti i buoni in un PDF) ⭐ CONSIGLIATO PER STAMPA
2. Word separati (ZIP con tutti i .docx singoli)

Inserisci il numero (1 o 2): 1

📄 Creazione PDF unico...

📦 Verifica LibreOffice...
✅ Presente

🔄 Conversione Word → PDF con LibreOffice...

  [1/12] Buono_27a56.docx... ✓
  [2/12] Buono_28349.docx... ✓
  [3/12] Buono_293b3.docx... ✓
  [4/12] Buono_4ceb9.docx... ✓
  [5/12] Buono_7ad9f.docx... ✓
  [6/12] Buono_86f7b.docx... ✓
  [7/12] Buono_8844d.docx... ✓
  [8/12] Buono_c2400.docx... ✓
  [9/12] Buono_df6c5.docx... ✓
  [10/12] Buono_dfb11.docx... ✓
  [11/12] Buono_e1aad.docx... ✓
  [12/12] Buono_fc743.docx... ✓

📦 Installazione PyPDF2...
📑 Unione di 12 PDF in un unico file...

✅ PDF creato: buoni_carburante_COMPLETI.pdf
   Dimensione: 3682.3 KB
   Pagine: 24
   Buoni: 12

⬇️ Download automatico...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 PDF PRONTO PER LA STAMPA!
   Tutti i buoni in sequenza con QR corretti

✅ OPERAZIONE COMPLETATA!

💡 SUGGERIMENTI:
   • Apri il PDF e stampa tutto
   • I QR sono unici e corretti per ogni buono

***************************************************************************
⭐ Se hai bisogno di altri buoni, parti dalla CELLA 2, non c'è bisogno di cancellare gli output! ⭐
***************************************************************************
